# 말뭉치를 다른 임베딩 모델로 인코딩 (Colab)

**하는 일은 하나입니다** — 논문 37,854편을 고른 모델로 인코딩해 `.npy` 하나를 만듭니다.
판정은 여기서 하지 않습니다. 행렬을 저장소로 가져가면 기존 하네스가 잽니다.

## 넘어야 하는 바

미래 라벨 과제의 **간선 0개 집단**(1,112 시드, 그래프가 0점이라 임베딩이 유일한 신호)에서
`text-embedding-3-large`를 **구간이 0을 제외하도록** 이겨야 합니다. 같은 자리에서 작은
형제 `bge-base-en-v1.5`가 그 선 아래로 측정됐습니다(recall@5 −0.0102 [−0.0159, −0.0048]).
빈 바가 아닙니다 — ADR-039.

| 모델 | 크기 | 차원 | 여기서 재는가 |
|---|---:|---:|---|
| `BAAI/bge-m3` | 568M | 1024 | 예 |
| `Qwen/Qwen3-Embedding-0.6B` | 595M | 1024 | 예 |
| `Qwen/Qwen3-Embedding-8B` | 8B | 4096 | **아니오** — fp16 16GB라 배포 경로가 없습니다 |

## 하지 않는 것

**텍스트를 다시 만들지 않습니다.** 순서와 문자열이 `corpus.jsonl.gz`에 굳어 있고, 다시
만들면 재는 것이 모델이 아니라 **전처리 차이**가 됩니다.

**Aura를 건드리지 않습니다.** 비교는 파생 캐시에서 돌아가므로 인덱스 재생성도 비용도 없습니다.


## 1. 설정 — 여기만 바꿉니다


In [ ]:
# 이 셀만 바꾸고 나머지는 그대로 실행하세요.
MODEL = "BAAI/bge-m3"            # 또는 "Qwen/Qwen3-Embedding-0.6B"
BATCH = 16                        # OOM이 나면 8로 줄이세요

# 말뭉치를 어디서 받을지. HF 데이터셋 저장소에 올려 뒀다면 USE_HF=True로 두고,
# 아니면 왼쪽 파일 창에 corpus.jsonl.gz를 직접 올리세요(18MB).
USE_HF = False
DATA_REPO = "arxiv-graphrag-colab"   # USE_HF=True일 때만 씁니다
CORPUS = "corpus.jsonl.gz"

# SMOKE=True면 200편만 인코딩해 코드가 도는지 봅니다(CPU로도 1분).
SMOKE = False


## 2. 환경 확인

GPU가 없으면 멈춥니다. CPU로는 이 크기 모델이 10시간을 넘겨서, 시작하는 것 자체가
낭비입니다(이 프로젝트 PC에서 109M 모델이 CPU로 3.5시간이었습니다).


In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None or not torch.cuda.is_available():
    print('GPU가 없습니다. 런타임 > 런타임 유형 변경에서 T4를 고르세요.')
    if not SMOKE:
        raise SystemExit('GPU 없이 전체 인코딩을 시작하지 않습니다.')
else:
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'{name}  {total:.1f}GB')


In [ ]:
%pip -q install -U sentence-transformers
print('설치 완료')


## 3. 말뭉치 받기

저장소에서 `uv run python -m models.embedding_tuning.export_for_colab`(32초)로 만든
`exports/colab/corpus.jsonl.gz`입니다. 37,854행이고 **파생 캐시와 같은 순서**입니다.


In [ ]:
import gzip
import json
from pathlib import Path

path = Path(CORPUS)
if USE_HF and not path.exists():
    from google.colab import userdata
    from huggingface_hub import HfApi, hf_hub_download
    token = userdata.get('HF_TOKEN')
    owner = HfApi(token=token).whoami()['name']
    path = Path(hf_hub_download(f'{owner}/{DATA_REPO}', CORPUS,
                                repo_type='dataset', token=token))

if not path.exists():
    raise SystemExit(f'{CORPUS}가 없습니다. 왼쪽 파일 창에 올리거나 USE_HF=True로 두세요.')

with gzip.open(path, 'rt', encoding='utf-8') as handle:
    rows = [json.loads(line) for line in handle]

if SMOKE:
    rows = rows[:200]

texts = [row['text'] for row in rows]
print(f'{len(texts)}편, 첫 줄: {texts[0][:80]}...')


## 4. 인코딩

`normalize_embeddings=True`와 `float32`는 기존 비교와 **같아야** 합니다
(`models/embedding_tuning/compare_models.py`). 바꾸면 비교가 깨집니다.

BGE가 권하는 질의 접두사는 **쓰지 않습니다** — 그 접두사는 짧은 질의로 긴 문서를 찾는
비대칭 검색용이고, 이 과제는 논문으로 논문을 찾는 대칭 검색입니다.

중간에 끊겨도 처음부터 하지 않도록 2,000편씩 조각으로 저장합니다. 다시 실행하면
남은 조각부터 이어서 합니다.


In [ ]:
import re
import time

import numpy as np
from sentence_transformers import SentenceTransformer

slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-')
out = Path(f'local-{slug}.npy')
chunks = Path(f'chunks-{slug}')
chunks.mkdir(exist_ok=True)
STEP = 2000

model = SentenceTransformer(MODEL, trust_remote_code=True)
print('max_seq_length', model.max_seq_length)

start = time.perf_counter()
for begin in range(0, len(texts), STEP):
    part = chunks / f'{begin:06d}.npy'
    if part.exists():
        continue  # 끊겼다가 다시 돌린 경우
    vectors = model.encode(
        texts[begin:begin + STEP], batch_size=BATCH,
        normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True,
    )
    np.save(part, np.asarray(vectors, dtype=np.float32))
    done = min(begin + STEP, len(texts))
    print(f'{done}/{len(texts)}  {(time.perf_counter() - start) / 60:.1f}분', flush=True)

matrix = np.concatenate([np.load(part) for part in sorted(chunks.glob('*.npy'))])
print('행렬', matrix.shape, matrix.dtype)


## 5. 저장하고 확인

행 수가 말뭉치와 다르면 저장하지 않습니다 — 어긋난 행렬을 가져가면 **조용히 다른**
**논문의 벡터로** 점수가 나옵니다. 저장소 쪽 `Graph`도 같은 검사를 합니다.


In [ ]:
if matrix.shape[0] != len(texts):
    raise SystemExit(f'행 수가 다릅니다: {matrix.shape[0]} != {len(texts)}')

norms = np.linalg.norm(matrix, axis=1)
print(f'노름 중앙값 {float(np.median(norms)):.4f} (정규화됐으면 1.0)')

np.save(out, matrix)
card = {
    'model': MODEL,
    'rows': int(matrix.shape[0]),
    'dimensions': int(matrix.shape[1]),
    'max_seq_length': int(model.max_seq_length),
    'normalize_embeddings': True,
    'minutes': round((time.perf_counter() - start) / 60, 1),
    'corpus': CORPUS,
    'smoke': SMOKE,
}
Path(out.stem + '.json').write_text(json.dumps(card, ensure_ascii=False, indent=2))
print(json.dumps(card, ensure_ascii=False, indent=2))
print(f'{out}  {out.stat().st_size / 1e6:.0f}MB')


## 6. 내려받기

`.npy`와 `.json`을 저장소의 `derived/2026-10-14/`에 두고 아래를 돌리면 판정이 나옵니다.

```bash
uv run python -m evals.related_study --cases evals/cases_related_future.json \
    --variants vector --embeddings derived/2026-10-14/local-<slug>.npy \
    --output reports/embedding_<name>.json

uv run python -m evals.embedding_slice_study \
    --baseline reports/embedding_openai.json --candidate reports/embedding_<name>.json
```


In [ ]:
if not SMOKE:
    try:
        from google.colab import files
        files.download(str(out))
        files.download(out.stem + '.json')
    except Exception as exc:
        print('자동 내려받기가 안 되면 왼쪽 파일 창에서 직접 받으세요:', exc)
